In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [27]:
import sys
import os

sys.path.append(os.path.abspath("..")) 

In [ ]:
import torch
import torch

def _patched_solve(B, A):
    return torch.linalg.solve(A, B), None

torch.solve = _patched_solve

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from losses import TimePointOverallLoss
from timepoint import TimePointModel  # your encoder+decoders combined
from synthetic_data.timepoint_dataset import TimePointDataset

In [29]:
def collate_fn(batch):
    return {
        "x": torch.stack([b["x"] for b in batch]),
        "x_w": torch.stack([b["x_w"] for b in batch]),
        "kp": torch.stack([b["kp"] for b in batch]),
        "kp_w": torch.stack([b["kp_w"] for b in batch]),
        "match_mask": torch.stack([b["match_mask"] for b in batch])
    }

In [34]:
from data_loader import NPZLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

from torch.utils.data import DataLoader

dataset = NPZLoader(
    "../data_cpab",
    use_signal="abp"   # or "cbfv"
)

loader = DataLoader(
    dataset,
    batch_size=2,     # ⚠️ keep small (match_mask!)
    shuffle=True,
    collate_fn=collate_fn
)

model = TimePointModel()
model = model.to(device)

model = TimePointModel().to(device)

criterion = TimePointOverallLoss(
    mp=1.0,
    mn=0.1,
    lambda_desc=1.0
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

device: cpu


TypeError: DescriptorDecoder.__init__() got an unexpected keyword argument 'cell_size'

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        x = batch["x"].to(device).unsqueeze(1)     # [B,1,L]
        x_w = batch["x_w"].to(device).unsqueeze(1)

        kp = batch["kp"].to(device)
        kp_w = batch["kp_w"].to(device)

        match_mask = batch["match_mask"].to(device)

        # --- forward ---
        S_logits, D = model(x)
        S_w_logits, D_w = model(x_w)

        # --- loss ---
        loss, loss_dict = criterion(
            S_logits, kp,
            S_w_logits, kp_w,
            D, D_w,
            match_mask
        )

        # --- backward ---
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"\nEpoch {epoch}")
    print(f"Total loss: {total_loss:.4f}")
    print(loss_dict)

In [ ]:
def visualize(sample, model):
    model.eval()

    x = sample["x"].unsqueeze(0).unsqueeze(0).to(device)
    x_w = sample["x_w"].unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        S_logits, _ = model(x)
        S_w_logits, _ = model(x_w)

    S = torch.sigmoid(S_logits).cpu().numpy()[0]
    S_w = torch.sigmoid(S_w_logits).cpu().numpy()[0]

    t = np.arange(len(S))

    plt.figure(figsize=(12,5))
    plt.plot(t, sample["x"], label="ABP")
    plt.plot(t, sample["x_w"], label="Warped", alpha=0.7)

    plt.scatter(t[S > 0.5], sample["x"][S > 0.5], color="red", label="KP")
    plt.scatter(t[S_w > 0.5], sample["x_w"][S_w > 0.5], color="blue", label="KP warped")

    plt.legend()
    plt.title("Predicted keypoints")
    plt.show()

In [ ]:
sample = dataset[0]
visualize(sample, model)